In [1]:
# ======================================================
# Stage 1 – Metadata Collector
# Generates metadata_stage1_all.csv
# ======================================================
import os, csv

# --- 1. set your base folder (change path if needed)
base_dir = r"C:\Users\nabal\Documents\FYP\chosen data"
output_csv = os.path.join(os.path.dirname(base_dir), "metadata_stage1_all.csv")

# --- 2. list of maqams to include
maqams = ["Hijaz", "Nahawand", "Saba"]

rows = []
for maqam in maqams:
    folder = os.path.join(base_dir, maqam)
    if not os.path.exists(folder):
        print(f"⚠️ folder not found: {folder}")
        continue

    for f in sorted(os.listdir(folder)):
        if f.lower().endswith(".wav"):
            rows.append({
                "maqam": maqam,
                "file_name": f,
                "reciter": "Unknown",   # <-- you’ll fill this manually
                "src_path": os.path.join(folder, f)
            })

# --- 3. write to CSV
os.makedirs(os.path.dirname(output_csv), exist_ok=True)
with open(output_csv, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=["maqam", "file_name", "reciter", "src_path"])
    writer.writeheader()
    writer.writerows(rows)

print(f"✅ Metadata file created: {output_csv}")
print(f"Total files listed: {len(rows)}")


✅ Metadata file created: C:\Users\nabal\Documents\FYP\metadata_stage1_all.csv
Total files listed: 171


In [2]:
# ============================
# Verify the MANUALLY balanced set
# ============================
import os, pandas as pd

BASE = r"C:\Users\nabal\Documents\FYP"
BAL  = os.path.join(BASE, "chosen data")
MAQAMS = ["Hijaz", "Nahawand", "Saba"]

# If you still have metadata_stage1_all.csv with reciters filled, reuse it:
meta_all_path = os.path.join(BASE, "metadata_stage1_all.csv")
meta_all = pd.read_csv(meta_all_path)

# Keep only files that actually exist in data_balanced
rows = []
for maq in MAQAMS:
    folder = os.path.join(BAL, maq)
    for f in os.listdir(folder):
        if f.lower().endswith(".wav"):
            # find reciter from your labelled metadata
            r = meta_all.loc[
                (meta_all["maqam"]==maq) & (meta_all["file_name"]==f),
                "reciter"
            ]
            rec = r.iloc[0] if len(r) else "Unknown"
            rows.append({"maqam": maq, "file_name": f, "reciter": rec,
                         "dst_path": os.path.join(folder, f)})

meta_bal = pd.DataFrame(rows).sort_values(["maqam","reciter","file_name"]).reset_index(drop=True)

# Save a manifest of the final balanced set
out_bal_csv = os.path.join(BASE, "metadata_stage1_balanced.csv")
meta_bal.to_csv(out_bal_csv, index=False)
print("✅ Saved balanced manifest:", out_bal_csv)

# Counts per maqam and per (maqam, reciter)
c1 = meta_bal.groupby("maqam").size().rename("count").reset_index()
c2 = meta_bal.groupby(["maqam","reciter"]).size().rename("count").reset_index()

c1.to_csv(os.path.join(BASE, "counts_by_maqam_post.csv"), index=False)
c2.to_csv(os.path.join(BASE, "counts_by_maqam_reciter_post.csv"), index=False)

print("\nFinal counts by maqam:\n", c1)
print("\nFinal counts by maqam & reciter:\n", c2)


✅ Saved balanced manifest: C:\Users\nabal\Documents\FYP\metadata_stage1_balanced.csv

Final counts by maqam:
       maqam  count
0     Hijaz     57
1  Nahawand     57
2      Saba     57

Final counts by maqam & reciter:
        maqam reciter  count
0      Hijaz       A     44
1      Hijaz       B      7
2      Hijaz       C      6
3   Nahawand       D      4
4   Nahawand       E      9
5   Nahawand       F      6
6   Nahawand       G      9
7   Nahawand       H      1
8   Nahawand       I      1
9   Nahawand       J     10
10  Nahawand       K     11
11  Nahawand       L      6
12      Saba       M     31
13      Saba       N      5
14      Saba       O      6
15      Saba       P      7
16      Saba       Q      4
17      Saba       R      4


In [4]:
import librosa, os

root = r"C:\Users\nabal\Documents\FYP\cleaned_chosen data"
for root, _, files in os.walk(root):
    for f in files:
        if f.endswith(".wav"):
            path = os.path.join(root, f)
            y, sr = librosa.load(path, sr=None, mono=True)
            print(f"{f}: sr={sr}, len={len(y)/sr:.1f}s, mean_amp={abs(y).mean():.4f}")


Hijaz_00.wav: sr=22050, len=35.3s, mean_amp=0.0349
Hijaz_01.wav: sr=22050, len=29.9s, mean_amp=0.0357
Hijaz_02.wav: sr=22050, len=35.2s, mean_amp=0.0353
Hijaz_03.wav: sr=22050, len=58.0s, mean_amp=0.0357
Hijaz_04.wav: sr=22050, len=45.0s, mean_amp=0.0359
Hijaz_05.wav: sr=22050, len=58.1s, mean_amp=0.0351
Hijaz_06.wav: sr=22050, len=43.3s, mean_amp=0.0349
Hijaz_07.wav: sr=22050, len=49.3s, mean_amp=0.0359
Hijaz_08.wav: sr=22050, len=37.0s, mean_amp=0.0362
Hijaz_09.wav: sr=22050, len=46.5s, mean_amp=0.0339
Hijaz_10.wav: sr=22050, len=46.2s, mean_amp=0.0353
Hijaz_11.wav: sr=22050, len=32.9s, mean_amp=0.0353
Hijaz_12.wav: sr=22050, len=47.4s, mean_amp=0.0349
Hijaz_13.wav: sr=22050, len=42.2s, mean_amp=0.0360
Hijaz_14.wav: sr=22050, len=53.0s, mean_amp=0.0353
Hijaz_15.wav: sr=22050, len=31.0s, mean_amp=0.0356
Hijaz_16.wav: sr=22050, len=31.2s, mean_amp=0.0354
Hijaz_17.wav: sr=22050, len=31.7s, mean_amp=0.0348
Hijaz_18.wav: sr=22050, len=59.2s, mean_amp=0.0350
Hijaz_19.wav: sr=22050, len=38.

In [5]:
import librosa, numpy as np, glob, os, pandas as pd
root = r"C:\Users\nabal\Documents\FYP\cleaned_chosen data"
rows=[]
for maqam in ["Hijaz","Nahawand","Saba"]:
    for f in glob.glob(os.path.join(root, maqam, "*.wav")):
        y, sr = librosa.load(f, sr=None, mono=True)
        rows.append({"maqam": maqam, "file": os.path.basename(f), "mean_amp": float(np.mean(np.abs(y)))})
df = pd.DataFrame(rows)
print(df.groupby("maqam")["mean_amp"].agg(["mean","std","min","max","count"]).round(4))


            mean     std     min     max  count
maqam                                          
Hijaz     0.0346  0.0013  0.0308  0.0372     57
Nahawand  0.0348  0.0021  0.0303  0.0393     57
Saba      0.0341  0.0031  0.0298  0.0409     57
